In [1]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv('../data/other_data/prem_2425_pressing_injuries.csv')
df

,name,played,pressed seqs,ppda,start distance (m),total,shot ending,goal ending,% end in shot,Number of injuries leading to 1+ games missed,Days lost
0,Arsenal,38,492,10.0,44.7,335,48,6,0.1433,36,1297
1,Aston Villa,38,410,13.1,40.5,237,42,5,0.1772,35,804
2,Bournemouth,38,480,9.9,43.3,337,68,10,0.2018,23,1185
3,Brentford,38,459,11.9,41.2,319,45,6,0.1411,26,1027
4,Brighton,38,493,10.8,44.1,325,55,3,0.1692,48,1944
5,Chelsea,38,411,12.0,42.1,295,53,6,0.1797,24,828
6,Crystal Palace,38,430,13.7,41.3,252,46,6,0.1825,23,1041
7,Everton,38,420,14.9,41.8,266,38,4,0.1429,24,1120
8,Fulham,38,372,13.0,40.3,207,37,4,0.1787,24,851
9,Ipswich,38,353,14.9,39.1,234,43,1,0.1838,41,1506


In [3]:
df.columns

Index(['name', 'played', 'pressed seqs', 'ppda', 'start distance (m)', 'total',
       'shot ending', 'goal ending', '% end in shot',
       'Number of injuries leading to 1+ games missed', 'Days lost'],
      dtype='object')

In [14]:
fig = px.scatter(
    df,
    x="pressed seqs",
    y="Days lost",
    text="name",  # Add team names as labels
    trendline="ols"
)
fig.update_traces(textposition='top center')  # Position the labels
fig.show()

In [7]:
import plotly.figure_factory as ff

# Select only numeric columns for correlation
corr = df.select_dtypes(include='number').corr()

z_rounded = corr.values.round(2)

fig = ff.create_annotated_heatmap(
    z=corr.values,
    x=list(corr.columns),
    y=list(corr.index),
    annotation_text=z_rounded,  # Show rounded values as annotations
    colorscale='Viridis',
    showscale=True
)
fig.update_layout(
    title="Correlation Heatmap",
    width=800,
    height=600
)
fig.write_html("correlation_heatmap.html") 
fig.show()

In [12]:
import numpy as np

# Get the pressed seqs values
tottenham = df[df['name'] == 'Tottenham'].iloc[0]
brentford = df[df['name'] == 'Brentford'].iloc[0]
brentford_pressed = brentford['pressed seqs']

# Linear regression: pressed seqs vs injuries, days lost, goal ending
for target in ['Number of injuries leading to 1+ games missed', 'Days lost', 'goal ending']:
    slope, intercept = np.polyfit(df['pressed seqs'], df[target], 1)
    predicted = slope * brentford_pressed + intercept
    print(f"If Tottenham had Brentford's pressed seqs ({brentford_pressed}), predicted {target}: {predicted:.1f} (was {tottenham[target]})")

# Optional: show the difference
print("\nChange in values for Tottenham if they pressed like Brentford:")
for target in ['Number of injuries leading to 1+ games missed', 'Days lost', 'goal ending']:
    slope, intercept = np.polyfit(df['pressed seqs'], df[target], 1)
    predicted = slope * brentford_pressed + intercept
    change = predicted - tottenham[target]
    print(f"{target}: {change:+.1f}")

If Tottenham had Brentford's pressed seqs (459), predicted Number of injuries leading to 1+ games missed: 30.7 (was 41)
If Tottenham had Brentford's pressed seqs (459), predicted Days lost: 1168.0 (was 1553)
If Tottenham had Brentford's pressed seqs (459), predicted goal ending: 6.5 (was 7)

Change in values for Tottenham if they pressed like Brentford:
Number of injuries leading to 1+ games missed: -10.3
Days lost: -385.0
goal ending: -0.5
